In [11]:
import pandas as pd
import io
from nba_manim.players import all_players
import requests
from nba_manim.data import transforms
import time

In [3]:
df_players = all_players()
df_players['From'] = pd.to_numeric(df_players['From'], errors='coerce')

In [8]:
df_players2000 = df_players[df_players["From"]>2000]
BASE_URL = "https://www.basketball-reference.com/"
params = None
headers = {"User-Agent": "Mozilla/5.0 (personal research script)"}

In [16]:
df_salaries = pd.DataFrame()
failed_players = []
for player_id in df_players2000['bref_id']:
    player_name = df_players2000[df_players2000['bref_id']==player_id]['Player'].values[0]
    print(player_id, player_name)
    url = f'{BASE_URL}/players/{player_id[0]}/{player_id}.html'

    try:
        resp = requests.get(url, params=params, headers=headers, timeout=15)
        resp.raise_for_status()
    except requests.exceptions.HTTPError as e:
        print(f'  -> page missing for {player_id}: {e}')
        failed_players.append({'bref_id': player_id, 'reason': 'http_error', 'detail': str(e)})
        time.sleep(8)
        continue

    resp.encoding = resp.apparent_encoding or 'utf-8'   # avoid the Ã³-style mojibake
    html = resp.text.replace("<!--", "").replace("-->", "")

    try:
        tables = pd.read_html(io.StringIO(html), attrs={"id": "all_salaries"}, flavor="lxml")
    except ValueError:
        print(f'  -> no salary table for {player_id}, skipping')
        failed_players.append({'bref_id': player_id, 'reason': 'no_salary_table', 'detail': None})
        time.sleep(8)
        continue

    for table in tables:
        table["Player"] = player_id
        df_salaries = pd.concat([df_salaries, table], ignore_index=True)
    time.sleep(8)

df_salaries['Salary_int'] = df_salaries['Salary'].apply(
    lambda x: transforms.parse_dollar_to_int(x) if isinstance(x, str) else x
)

print(f"{len(failed_players)} player(s) had no usable data:")
print(pd.DataFrame(failed_players))

abrinal01 Ãlex Abrines
achiupr01 Precious Achiuwa
ackeral01 Alex Acker
acyqu01 Quincy Acy
adamsha01 Hassan Adams
adamsja01 Jaylen Adams
adamsjo01 Jordan Adams
adamsst01 Steven Adams
adebaba01 Bam Adebayo
adelde01 Deng Adel
adrieje01 Jeff Adrien
afflaar01 Arron Afflalo
agbajoc01 Ochai Agbaji
agerma01 Maurice Ager
ahearbl01 Blake Ahearn
ajincal01 Alexis AjinÃ§a
akognjo01 Josh Akognon
akoonde01 DeVaughn Akoon-Purcell
  -> no salary table for akoonde01, skipping
alabiso01 Solomon Alabi
aldamsa01 Santi Aldama
aldemfu01 Furkan Aldemir
aldrico01 Cole Aldrich
aldrila01 LaMarcus Aldridge
alexacl01 Cliff Alexander
alexaco02 Courtney Alexander
alexajo01 Joe Alexander
alexaky01 Kyle Alexander
alexatr01 Trey Alexander
alexaty01 Ty-Shon Alexander
alexani01 Nickeil Alexander-Walker
alkinra01 Rawle Alkins
allengr01 Grayson Allen
allenja01 Jarrett Allen
allenka01 Kadeem Allen
allenla01 Lavoy Allen
allenma01 Malik Allen
allenti01 Timmy Allen
allento01 Tony Allen
allrela01 Lance Allred
  -> no salary ta

ConnectionError: HTTPSConnectionPool(host='www.basketball-reference.com', port=443): Max retries exceeded with url: //players/g/gainesu01.html (Caused by NameResolutionError("HTTPSConnection(host='www.basketball-reference.com', port=443): Failed to resolve 'www.basketball-reference.com' ([Errno 11001] getaddrinfo failed)"))

In [10]:
df_salaries

,Season,Team,Lg,Salary,Player,Salary_int
0,2016-17,Oklahoma City Thunder,NBA,"$5,994,764",abrinal01,5994764
1,2017-18,Oklahoma City Thunder,NBA,"$5,725,000",abrinal01,5725000
2,2018-19,Oklahoma City Thunder,NBA,"$3,575,183",abrinal01,3575183
3,Career,(may be incomplete),NaN,"$15,294,947",abrinal01,15294947
4,2020-21,Miami Heat,NBA,"$2,582,160",achiupr01,2582160
5,2021-22,Toronto Raptors,NBA,"$2,711,280",achiupr01,2711280
6,2022-23,Toronto Raptors,NBA,"$2,840,160",achiupr01,2840160
7,2023-24,New York Knicks,NBA,"$4,379,527",achiupr01,4379527
8,2024-25,New York Knicks,NBA,"$6,000,000",achiupr01,6000000
9,2025-26,Sacramento Kings,NBA,"$2,111,516",achiupr01,2111516


In [38]:
12_000*2

24000